In [1]:
from pynq import Overlay
import time

ol = Overlay("posterization_filter.bit")

# Get IP and set control register
# ip = ol.video.invstripe_0
ip = ol.video.cartoon_filter_0
rm = ip.register_map
rm.edge_threshold = 100
rm.bright_factor = 320
rm.quant_levels = 5

# Access HDMI
hdmi_in = ol.video.hdmi_in
hdmi_out = ol.video.hdmi_out

print("Configuring HDMI_IN...")
hdmi_in.configure()
hdmi_in.start()
print("HDMI_IN mode:", hdmi_in.mode)

print("Configuring HDMI_OUT to match HDMI_IN mode...")
hdmi_out.configure(hdmi_in.mode)
hdmi_out.start()
print("HDMI_OUT mode:", hdmi_out.mode)

print("HDMI pipeline is running. Output should appear on the monitor.")

print("HDMI running. You should see output on the monitor.")
print("Pumping frames... Interrupt the cell (or Ctrl+C) to stop.")

try:
    while True:
        frame = hdmi_in.readframe()    # blocks until a new frame arrives
        hdmi_out.writeframe(frame)     # sends it to output (your IP is in HW path)
        print("kkkkk")
        # no sleep needed; rate is determined by HDMI source
except KeyboardInterrupt:
    print("Interrupted.")
finally:
    print("Stopping HDMI...")
    try:
        hdmi_in.stop()
    except Exception as e:
        print("hdmi_in.stop() error:", e)
    try:
        hdmi_out.stop()
    except Exception as e:
        print("hdmi_out.stop() error:", e)
        
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("Interrupted.")
        
    print("Stopping HDMI...")
    try:
        hdmi_in.stop()
    except Exception as e:
        print("hdmi_in.stop() error:", e)

    try:
        hdmi_out.stop()
    except Exception as e:
        print("hdmi_out.stop() error:", e)

    print("Done.")

Configuring HDMI_IN...
HDMI_IN mode: VideoMode: width=1280 height=720 bpp=24
Configuring HDMI_OUT to match HDMI_IN mode...
HDMI_OUT mode: VideoMode: width=1280 height=720 bpp=24
HDMI pipeline is running. Output should appear on the monitor.
HDMI running. You should see output on the monitor.
Pumping frames... Interrupt the cell (or Ctrl+C) to stop.
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
kkkkk
k

shape: (720, 1280, 3) dtype: uint8
min/max: 0 255


HDMI running. You should see output on the monitor.
Pumping frames... Interrupt the cell (or Ctrl+C) to stop.


Running... (Interrupt the cell to stop)
Interrupted.


Stopping HDMI...
Done.


In [3]:
from pynq import Overlay
import time
import numpy as np
import tracemalloc

tracemalloc.start()

# Configuration
WARMUP_FRAMES = 50
TEST_FRAMES = 2000

ol = Overlay("posterization_filter.bit")

# Get IP and set control register
ip = ol.video.cartoon_filter_0
rm = ip.register_map
rm.edge_threshold = 100
rm.bright_factor = 320
rm.quant_levels = 5

# Access HDMI
hdmi_in = ol.video.hdmi_in
hdmi_out = ol.video.hdmi_out

print("Configuring HDMI_IN...")
hdmi_in.configure()
hdmi_in.start()
print("HDMI_IN mode:", hdmi_in.mode)

print("Configuring HDMI_OUT to match HDMI_IN mode...")
hdmi_out.configure(hdmi_in.mode)
hdmi_out.start()
print("HDMI_OUT mode:", hdmi_out.mode)

print("HDMI pipeline is running. Output should appear on the monitor.")

try:
    # Warmup phase (optional)
    print(f"\nWarming up with {WARMUP_FRAMES} frames...")
    for i in range(WARMUP_FRAMES):
        frame = hdmi_in.readframe()
        hdmi_out.writeframe(frame)
    
    # Benchmark phase
    print(f"\nBenchmarking latency for {TEST_FRAMES} frames...")
    times_ms = []
    
    for i in range(TEST_FRAMES):
        t0 = time.perf_counter()
        frame = hdmi_in.readframe()    # blocks until a new frame arrives
        hdmi_out.writeframe(frame)     # sends it to output (your IP is in HW path)
        t1 = time.perf_counter()
        
        times_ms.append((t1 - t0) * 1000.0)
        
        # Progress indicator
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{TEST_FRAMES} frames done")
    
    # ---- Results ----
    t = np.array(times_ms, dtype=np.float64)
    mean_ms = float(t.mean())
    p95_ms = float(np.percentile(t, 95))
    max_ms = float(t.max())
    min_ms = float(t.min())
    fps_equiv = 1000.0 / mean_ms if mean_ms > 0 else 0.0
    
    print("\n=== Hardware Accelerator Performance (HDMI pipeline) ===")
    print(f"Frames tested: {len(t)}")
    print(f"Mean: {mean_ms:.2f} ms/frame")
    print(f"Min : {min_ms:.2f} ms/frame")
    print(f"P95 : {p95_ms:.2f} ms/frame")
    print(f"Max : {max_ms:.2f} ms/frame")
    print(f"FPS equivalent (mean): {fps_equiv:.2f}")
    print(f"Meets 60fps budget (16.67 ms/frame)? {'YES' if mean_ms <= 16.67 else 'NO'}")
    
    # Optional: continue running indefinitely after benchmark
    print("\n--- Benchmark complete. Continue running? (Ctrl+C to stop) ---")
    while True:
        frame = hdmi_in.readframe()
        hdmi_out.writeframe(frame)
        
except KeyboardInterrupt:
    print("\nInterrupted.")
finally:
    print("Stopping HDMI...")
    try:
        hdmi_in.stop()
    except Exception as e:
        print("hdmi_in.stop() error:", e)
    try:
        hdmi_out.stop()
    except Exception as e:
        print("hdmi_out.stop() error:", e)
    print("Done.")

current, peak = tracemalloc.get_traced_memory()
print(f"Peak   : {peak / (1024**2):.1f} MB")
tracemalloc.stop()

Configuring HDMI_IN...
HDMI_IN mode: VideoMode: width=1280 height=720 bpp=24
Configuring HDMI_OUT to match HDMI_IN mode...
HDMI_OUT mode: VideoMode: width=1280 height=720 bpp=24
HDMI pipeline is running. Output should appear on the monitor.

Warming up with 50 frames...

Benchmarking latency for 2000 frames...
  50/2000 frames done
  100/2000 frames done
  150/2000 frames done
  200/2000 frames done
  250/2000 frames done
  300/2000 frames done
  350/2000 frames done
  400/2000 frames done
  450/2000 frames done
  500/2000 frames done
  550/2000 frames done
  600/2000 frames done
  650/2000 frames done
  700/2000 frames done
  750/2000 frames done
  800/2000 frames done
  850/2000 frames done
  900/2000 frames done
  950/2000 frames done
  1000/2000 frames done
  1050/2000 frames done
  1100/2000 frames done
  1150/2000 frames done
  1200/2000 frames done
  1250/2000 frames done
  1300/2000 frames done
  1350/2000 frames done
  1400/2000 frames done
  1450/2000 frames done
  1500/2000 

In [2]:
tracemalloc.start()
main()
current, peak = tracemalloc.get_traced_memory()
print(f"Peak   : {peak / (1024**2):.1f} MB")
tracemalloc.stop()

NameError: name 'tracemalloc' is not defined